# Create the shared multimodal hyperparameter tuning configurations

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import itertools
import json
import hashlib

import numpy as np
import pandas as pd


OUTPUT_DIR = (PROJECT_ROOT / 'experiments'/ '5_residual_hyper' / 'shared_search')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUTPUT_DIR / 'multimodal_tuning_configurations.csv'
JSON_PATH = OUTPUT_DIR / 'multimodal_tuning_configurations.json'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)

PROJECT_ROOT: /content/drive/MyDrive/dissertation
OUTPUT_DIR: /content/drive/MyDrive/dissertation/experiments/5_residual_hyper/shared_search


In [3]:
IMG_PROJ_DIM = 4

DROPOUTS = [0, 0.1, 0.3, 0.5]
LEARNING_RATES = [1e-3, 3e-4, 1e-4, 3e-5, 1e-5]
WEIGHT_DECAYS = [0, 1e-6, 1e-5, 1e-4]

# Current D3b configuration. This must be included.
ANCHOR = {
    'img_proj_dim': 4,
    'img_dropout': 0.3,
    'lr': 1e-4,
    'weight_decay': 1e-5,
}

SEARCH_BUDGET = 40
SEARCH_SEED = 0

full_space = [
    {
        'img_proj_dim': IMG_PROJ_DIM,
        'img_dropout': float(dropout),
        'lr': float(lr),
        'weight_decay': float(wd),
    }
    for dropout, lr, wd in itertools.product(
        DROPOUTS,
        LEARNING_RATES,
        WEIGHT_DECAYS,
    )
]

assert len(full_space) == 80
assert ANCHOR in full_space
assert 1 <= SEARCH_BUDGET <= len(full_space)

print('Full search space:', len(full_space))
print('Search budget:', SEARCH_BUDGET)
print('Anchor:', ANCHOR)

Full search space: 80
Search budget: 40
Anchor: {'img_proj_dim': 4, 'img_dropout': 0.3, 'lr': 0.0001, 'weight_decay': 1e-05}


In [4]:
def key(cfg):
    return (
        cfg['img_proj_dim'],
        cfg['img_dropout'],
        cfg['lr'],
        cfg['weight_decay'],
    )

remaining = [cfg for cfg in full_space if key(cfg) != key(ANCHOR)]

rng = np.random.default_rng(SEARCH_SEED)
sampled_indices = rng.choice(
    len(remaining),
    size=SEARCH_BUDGET - 1,
    replace=False,
)

sampled = [remaining[int(i)] for i in sampled_indices]
selected = [ANCHOR.copy()] + [cfg.copy() for cfg in sampled]

assert len(selected) == SEARCH_BUDGET
assert len({key(cfg) for cfg in selected}) == SEARCH_BUDGET
assert key(selected[0]) == key(ANCHOR)

for order, cfg in enumerate(selected, start=1):
    cfg['config_order'] = order
    cfg['config_id'] = f'cfg_{order:02d}'
    cfg['is_anchor'] = (order == 1)

selected_df = pd.DataFrame(selected)[
    [
        'config_order',
        'config_id',
        'is_anchor',
        'img_proj_dim',
        'img_dropout',
        'lr',
        'weight_decay',
    ]
]

display(selected_df)

,config_order,config_id,is_anchor,img_proj_dim,img_dropout,lr,weight_decay
0,1,cfg_01,True,4,0.3,0.00010,0.000010
1,2,cfg_02,False,4,0.1,0.00003,0.000000
2,3,cfg_03,False,4,0.5,0.00003,0.000010
3,4,cfg_04,False,4,0.1,0.00001,0.000001
4,5,cfg_05,False,4,0.3,0.00100,0.000000
5,6,cfg_06,False,4,0.5,0.00100,0.000001
6,7,cfg_07,False,4,0.5,0.00010,0.000001
7,8,cfg_08,False,4,0.1,0.00100,0.000001
8,9,cfg_09,False,4,0.0,0.00010,0.000100
9,10,cfg_10,False,4,0.5,0.00003,0.000100


In [5]:
print('Selected-value coverage:')
for column in ['img_dropout', 'lr', 'weight_decay']:
    counts = (
        selected_df[column]
        .value_counts()
        .sort_index()
    )
    print(f'\n{column}')
    print(counts.to_string())

assert set(selected_df['img_dropout']) == set(DROPOUTS)
assert set(selected_df['lr']) == set(LEARNING_RATES)
assert set(selected_df['weight_decay']) == set(WEIGHT_DECAYS)

print('\nAll candidate values are represented at least once.')

Selected-value coverage:

img_dropout
img_dropout
0.0    10
0.1    11
0.3    12
0.5     7

lr
lr
0.00001     8
0.00003    10
0.00010     6
0.00030     6
0.00100    10

weight_decay
weight_decay
0.000000    10
0.000001    12
0.000010    10
0.000100     8

All candidate values are represented at least once.


In [6]:
if CSV_PATH.exists() or JSON_PATH.exists():
    raise FileExistsError(
        'A shared tuning configuration file already exists.\n'
        f'CSV:  {CSV_PATH}\n'
        f'JSON: {JSON_PATH}\n\n'
        'Do not regenerate it after tuning begins. If this is an '
        'intentional pre-experiment redesign, delete the existing '
        'files manually and rerun this notebook.'
    )

selected_df.to_csv(CSV_PATH, index=False)

payload = {
    'design': {
        'architecture': 'frozen CGM+Time + image-only residual fusion',
        'img_proj_dim': IMG_PROJ_DIM,
        'full_space_size': len(full_space),
        'search_budget': SEARCH_BUDGET,
        'search_seed': SEARCH_SEED,
        'selection_method': (
            'anchor first + fixed random sample without replacement '
            'of 39 configurations from the remaining full Cartesian space'
        ),
        'same_order_for_all_encoders': True,
        'encoders': [
            'resnet18',
            'vit_b_16',
            'clip',
            'dinov2',
        ],
        'anchor': ANCHOR,
        'search_space': {
            'img_dropout': DROPOUTS,
            'lr': LEARNING_RATES,
            'weight_decay': WEIGHT_DECAYS,
        },
    },
    'configurations': selected,
}

with open(JSON_PATH, 'w') as f:
    json.dump(payload, f, indent=2)
sha256 = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()

print('Saved shared search files:')
print(' ', CSV_PATH)
print(' ', JSON_PATH)
print('\nCSV SHA256:')
print(sha256)

Saved shared search files:
  /content/drive/MyDrive/dissertation/experiments/5_residual_hyper/shared_search/multimodal_tuning_configurations.csv
  /content/drive/MyDrive/dissertation/experiments/5_residual_hyper/shared_search/multimodal_tuning_configurations.json

CSV SHA256:
32e220271d6157119aec540c96162224e751d9f49207f511dc6cb171c77434d3
